# Chronos-2 Inference Speed Profiling

- **Baseline** — `Chronos2Pipeline.predict()`: the standard high-level API with full dataset/DataLoader machinery

## 1. Config

In [1]:
import math
from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

import os
os.environ.setdefault("HOME", os.environ.get("USERPROFILE", str(Path.home())))

import torch
import numpy as np
from torch.utils.data import DataLoader

from chop.models.chronos2.pipeline import Chronos2Pipeline
from chronos.chronos2.dataset import Chronos2Dataset, DatasetMode

# ── Config ────────────────────────────────────────────────────────────────────
DEVICE           = 'cuda' if torch.cuda.is_available() else 'cpu'
SEQ_LEN          = 1440   # context length per time series
N_FEATURES       = 20     # number of variates per sample
FORECAST_HORIZON = 96     # prediction length
BATCH_SIZES      = [20, 40, 60, 80, 100]  # number of multivariate samples per batch
WARMUP_RUNS      = 3
BENCH_RUNS       = 5
MODEL_ID         = 'amazon/chronos-2'

print(f'Device: {DEVICE}')
print(f'Sequence length: {SEQ_LEN}')
print(f'Number of features: {N_FEATURES}')
print(f'Forecast horizon: {FORECAST_HORIZON}')

Device: cuda
Sequence length: 1440
Number of features: 20
Forecast horizon: 96


## 2. Load model

In [2]:
from chop.models import get_model

model = get_model("chronos-2", pretrained=True, model_id=MODEL_ID)
model = model.to(DEVICE).to(torch.bfloat16)
pipeline = Chronos2Pipeline(model=model)
model.eval()

OUTPUT_PATCH_SIZE = model.chronos_config.output_patch_size
NUM_OUTPUT_PATCHES = math.ceil(FORECAST_HORIZON / OUTPUT_PATCH_SIZE)
FUTURE_LEN = NUM_OUTPUT_PATCHES * OUTPUT_PATCH_SIZE

print(f'✓ Model loaded successfully from {MODEL_ID}')
print(f'  Model type:        {type(model).__name__}')
print(f'  Output patch size: {OUTPUT_PATCH_SIZE}')
print(f'  Num output patches for horizon={FORECAST_HORIZON}: {NUM_OUTPUT_PATCHES}')


✓ Model loaded successfully from amazon/chronos-2
  Model type:        Chronos2Model
  Output patch size: 16
  Num output patches for horizon=96: 6


# 3. Benchmark with fev bench


In [3]:
# Benchmark function 

import pandas as pd


def fev_bench_inference_time(model, model_name, task_configs, batch_sizes, output_dir="artifacts"):
    import fev

    # Define benchmark
    benchmark = fev.Benchmark.from_list(task_configs)

    # Run benchmark for each model and batch size
    summaries = []
    inference_times = {}
    print(f'\n=== Benchmarking model: {model_name} ===')

    # Create pipeline for the model
    pipeline = Chronos2Pipeline(model=model)

    for task in benchmark.tasks:
        print(f'\n--- Task: {task.task_name} ---')

        inference_times[task.task_name] = {}

        for batch_size in batch_sizes:
            predictions, inference_time = pipeline.predict_fev(task, batch_size)
            summary = task.evaluation_summary(predictions, model_name)
            summaries.append(summary)

            inference_times[task.task_name][batch_size] = inference_time
            
            inference_time_per_sample = inference_time / batch_size
            print(f'Batch size: {batch_size} | Inference time per sample: {inference_time_per_sample:.4f} sec')

    # ── Save results to CSV ───────────────────────────────────────────────────
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # Summaries CSV
    summaries_df = pd.DataFrame(summaries)
    summaries_path = Path(output_dir) / f"{model_name}_summaries.csv"
    summaries_df.to_csv(summaries_path, index=False)
    print(f'\nSaved summaries → {summaries_path}')

    # Inference times CSV  (rows = tasks, columns = batch sizes)
    inference_rows = []
    for task_name, batch_dict in inference_times.items():
        for batch_size, t in batch_dict.items():
            inference_rows.append({
                "task": task_name,
                "batch_size": batch_size,
                "inference_time_s": t,
                "inference_time_per_sample_s": t / batch_size,
            })
    inference_df = pd.DataFrame(inference_rows)
    inference_path = Path(output_dir) / f"{model_name}_inference_times.csv"
    inference_df.to_csv(inference_path, index=False)
    print(f'Saved inference times → {inference_path}')

    return summaries_df, inference_df


In [9]:
# Benchmark on baseline chronos2 
tasks_configs = [
    {
        "dataset_path": "autogluon/chronos_datasets",
        "dataset_config": "m4_hourly",
        "horizon": 24,
    },
]

BATCH_SIZES = [20, 40, 60, 80, 100]

fev_bench_inference_time(
    model=model,
    model_name="Baseline",
    task_configs=tasks_configs,
    batch_sizes=BATCH_SIZES,
)


=== Benchmarking model: Baseline ===

--- Task: m4_hourly ---
Batch size: 20 | Inference time per sample: 0.0456 sec
Batch size: 40 | Inference time per sample: 0.0161 sec
Batch size: 60 | Inference time per sample: 0.0116 sec
Batch size: 80 | Inference time per sample: 0.0087 sec
Batch size: 100 | Inference time per sample: 0.0079 sec

Saved summaries → artifacts/Baseline_summaries.csv
Saved inference times → artifacts/Baseline_inference_times.csv


(  model_name                dataset_path dataset_config  horizon  num_windows  \
 0   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 1   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 2   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 3   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 4   Baseline  autogluon/chronos_datasets      m4_hourly       24            1   
 
    initial_cutoff  window_step_size  min_context_length max_context_length  \
 0             -24                24                   1               None   
 1             -24                24                   1               None   
 2             -24                24                   1               None   
 3             -24                24                   1               None   
 4             -24                24                   1               None   
 
    seasonality  ... static_co

In [7]:
from chop.models import get_model
from chop.models.chronos2.layers import FusedQKV_MHA

model_fusedqkv = get_model("chronos-2", pretrained=True, model_id=MODEL_ID)
model_fusedqkv = model_fusedqkv.to(DEVICE).to(torch.bfloat16)

replaced_count = 0

# Walk through every module and its immediate children
for name, module in model_fusedqkv.named_modules():
    for child_name, child_module in module.named_children():
        # If the child is the original MHA class, swap it!
        if child_module.__class__.__name__ == "MHA":
            setattr(module, child_name, FusedQKV_MHA(child_module))
            replaced_count += 1

print(f"Surgery complete: Swapped {replaced_count} standard MHA modules for FusedQKV_MHA.")

Surgery complete: Swapped 24 standard MHA modules for FusedQKV_MHA.


In [11]:
# run the benchmark again with the modified model
fev_bench_inference_time(
    model=model_fusedqkv,
    model_name="FusedQKV",
    task_configs=tasks_configs,
    batch_sizes=BATCH_SIZES,
)


=== Benchmarking model: FusedQKV ===

--- Task: m4_hourly ---
Batch size: 20 | Inference time per sample: 0.0810 sec
Batch size: 40 | Inference time per sample: 0.0200 sec
Batch size: 60 | Inference time per sample: 0.0128 sec
Batch size: 80 | Inference time per sample: 0.0107 sec
Batch size: 100 | Inference time per sample: 0.0091 sec

Saved summaries → artifacts/FusedQKV_summaries.csv
Saved inference times → artifacts/FusedQKV_inference_times.csv


(  model_name                dataset_path dataset_config  horizon  num_windows  \
 0   FusedQKV  autogluon/chronos_datasets      m4_hourly       24            1   
 1   FusedQKV  autogluon/chronos_datasets      m4_hourly       24            1   
 2   FusedQKV  autogluon/chronos_datasets      m4_hourly       24            1   
 3   FusedQKV  autogluon/chronos_datasets      m4_hourly       24            1   
 4   FusedQKV  autogluon/chronos_datasets      m4_hourly       24            1   
 
    initial_cutoff  window_step_size  min_context_length max_context_length  \
 0             -24                24                   1               None   
 1             -24                24                   1               None   
 2             -24                24                   1               None   
 3             -24                24                   1               None   
 4             -24                24                   1               None   
 
    seasonality  ... static_co